In [19]:
pip install chembl_webresource_client


   ---------------------------------------- 0.0/55.2 kB ? eta -:--:--
   ---------------------------------------- 55.2/55.2 kB 1.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/70.2 kB ? eta -:--:--
   ---------------------------------------- 70.2/70.2 kB 3.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/73.1 kB ? eta -:--:--
   ---------------------------------------- 73.1/73.1 kB ? eta 0:00:00
   ---------------------------------------- 0.0/67.5 kB ? eta -:--:--
   ---------------------------------------- 67.5/67.5 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: attrs
    Found existing installation: attrs 23.1.0
    Uninstalling attrs-23.1.0:
      Successfully uninstalled attrs-23.1.0
Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd

In [3]:
protein = pd.read_csv("data/egfr_protein.csv")
rna = pd.read_csv("data/egfr_rna.csv")
full = pd.read_csv("data/egfr_full_merged.csv")
mutation = pd.read_csv("data/egfr_mutation.csv")
phospho = pd.read_csv("data/egfr_phospho.csv")

In [18]:
from chembl_webresource_client.new_client import new_client

activity = new_client.activity

res = activity.filter(
    target_chembl_id="CHEMBL203",  # EGFR
    molecule_chembl_id="CHEMBL1079742"
).filter(
    standard_type="IC50"
)

df = pd.DataFrame(res)

ModuleNotFoundError: No module named 'chembl_webresource_client'

In [15]:
#ligands
ligand_df = pd.DataFrame({
    "ligand": ["erlotinib", "gefitinib", "osimertinib"],
    "target": ["EGFR", "EGFR", "EGFR"],
    "IC50_nM": [5, 12, 1],
})

In [4]:
#standarizing patient ids/cohort (gets important for merging)
def clean_patient_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.N$", "", regex=True)
        .str.replace(r"\.T$", "", regex=True)
    )


for df in [protein, rna, full, phospho, mutation]:
    if "PATIENT_ID" in df.columns and "patient_id" not in df.columns:
        df.rename(columns={"PATIENT_ID": "patient_id"}, inplace=True)

for df in [protein, rna, full, phospho, mutation]:
    df["patient_id"] = clean_patient_id(df["patient_id"])


 
protein["cohort"] = "CPTAC"
protein["batch_domain"] = "CPTAC"

rna["cohort"] = "CPTAC"
rna["batch_domain"] = "CPTAC"

full["cohort"] = "CPTAC"
full["batch_domain"] = "CPTAC"

phospho["cohort"] = "CPTAC"
phospho["batch_domain"] = "CPTAC"

mutation["cohort"] = "TCGA"
mutation["batch_domain"] = "TCGA"
mutation["gene"] = "EGFR"

In [4]:
#print(full.head())
#print(mutation.head())

***MUTATION FEATURES***

In [ ]:
#expression for protein features
expression_protein_features = full[[
    "patient_id",
    "EGFR_PROTEIN",
    "EGFR_RNA",
    "EGFR_activity_mean",
    "cohort",
    "batch_domain"
]].copy()


#collapse to one row per patient by taking the mean of the features
expression_protein_features = (
    expression_protein_features
    .groupby("patient_id", as_index=False)
    .agg({
        "EGFR_PROTEIN": "mean",
        "EGFR_RNA": "mean",
        "EGFR_activity_mean": "mean",
        "cohort": "first",
        "batch_domain": "first"
    })
)
#needto drop dupolicates
dup_counts = expression_protein_features["patient_id"].value_counts()
print("Patients with duplicate rows in expression_protein_features:", #instead of guessing (took a lot longer than expected to figure our >_<)
      (dup_counts > 1).sum())



expression_protein_features.head()

Patients with duplicate rows in expression_protein_features: 0


,patient_id,EGFR_PROTEIN,EGFR_RNA,EGFR_activity_mean,cohort,batch_domain
0,C3L-00001,26.919190,14.500,1.0,CPTAC,CPTAC
1,C3L-00009,25.188724,12.090,1.0,CPTAC,CPTAC
2,C3L-00080,25.203323,12.530,1.0,CPTAC,CPTAC
3,C3L-00083,25.336752,11.725,1.0,CPTAC,CPTAC
4,C3L-00093,24.736987,12.750,1.0,CPTAC,CPTAC


In [ ]:
print("Shape after deduplication:", expression_protein_features.shape)
print("Remaining duplicate patient_ids:",
      expression_protein_features["patient_id"].duplicated().sum()) #zero dupes!!

Shape after deduplication: (106, 6)
Remaining duplicate patient_ids: 0


***PHOSPHOSITES***

In [11]:
#phospho target sites
target_sites = ["Y1172", "Y1092", "Y1069", "Y1110", "Y1016"]
phospho_by_site = phospho[
    phospho["EGFR_binding_site"].isin(target_sites)
].copy()
#phospho_by_site.head()

egfr_expression_score = (phospho_by_site.groupby("patient_id", as_index=False)["EGFR_phospho_value"]
                        .mean()
                        .rename(columns={"EGFR_phospho_value": "EGFR_expression_score"})
)

In [12]:
phospho_model_data = (
    phospho_by_site
    .pivot_table(
        index="patient_id",
        columns="EGFR_binding_site",
        values="EGFR_phospho_value",
        aggfunc="mean"
    )
    .reset_index()
)


phospho_model_data.columns = [
    "patient_id"
] + [f"phospho_{c}" for c in phospho_model_data.columns[1:]]

In [ ]:
cptac_model_df = (
    expression_protein_features
    .merge(
        phospho_model_data,
        on="patient_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        egfr_expression_score,
        on="patient_id",
        how="left",
        validate="one_to_one"
    )
)

print("CPTAC model shape:", cptac_model_df.shape)
print("Duplicate patient_ids in cptac_model_df:",
      cptac_model_df["patient_id"].duplicated().sum())
print(cptac_model_df.head()) #need to fix the NanS

CPTAC model shape: (106, 12)
Duplicate patient_ids in cptac_model_df: 0
  patient_id  EGFR_PROTEIN  EGFR_RNA  EGFR_activity_mean cohort batch_domain  \
0  C3L-00001     26.919190    14.500                 1.0  CPTAC        CPTAC   
1  C3L-00009     25.188724    12.090                 1.0  CPTAC        CPTAC   
2  C3L-00080     25.203323    12.530                 1.0  CPTAC        CPTAC   
3  C3L-00083     25.336752    11.725                 1.0  CPTAC        CPTAC   
4  C3L-00093     24.736987    12.750                 1.0  CPTAC        CPTAC   

   phospho_Y1016  phospho_Y1069  phospho_Y1092  phospho_Y1110  phospho_Y1172  \
0            NaN            NaN            NaN            NaN      20.075879   
1            NaN            NaN      16.931486            NaN      18.767056   
2            NaN            NaN            NaN            NaN      18.026248   
3            NaN            NaN            NaN            NaN            NaN   
4            NaN            NaN            NaN 

In [8]:
#CPTAC modeling table

cptac_model_df = expression_protein_features.merge(
    phospho_model_data,
    on="patient_id",
    how="left"
).merge(
    egfr_expression_score,
    on="patient_id",
    how="left"
)

print("CPTAC model shape:", cptac_model_df.shape)
print(cptac_model_df.head())

CPTAC model shape: (207, 12)
  patient_id  EGFR_PROTEIN  EGFR_RNA  EGFR_activity_mean cohort batch_domain  \
0  C3L-00001     28.192875     17.07                   1  CPTAC        CPTAC   
1  C3L-00009     25.219585     11.86                   1  CPTAC        CPTAC   
2  C3L-00080     25.238803     12.58                   1  CPTAC        CPTAC   
3  C3L-00083     25.041583     10.94                   1  CPTAC        CPTAC   
4  C3L-00093     24.469511     12.78                   1  CPTAC        CPTAC   

   phospho_Y1016  phospho_Y1069  phospho_Y1092  phospho_Y1110  phospho_Y1172  \
0            NaN            NaN            NaN            NaN      20.075879   
1            NaN            NaN      16.931486            NaN      18.767056   
2            NaN            NaN            NaN            NaN      18.026248   
3            NaN            NaN            NaN            NaN            NaN   
4            NaN            NaN            NaN            NaN            NaN   

   EGFR_e

In [9]:
#clean mutation features
mutation["mutation"] = mutation["mutation"].astype(str).str.strip()
mutation["EGFR_type"] = mutation["EGFR_type"].astype(str).str.strip()

def label_egfr_hotspot(mutation_value: str) -> str:
    m = str(mutation_value).upper()
    if "EXON 19" in m or "19DEL" in m or "DEL19" in m:
        return "exon19del"
    elif "L858R" in m:
        return "L858R"
    elif "T790M" in m:
        return "T790M"
    elif "C797S" in m:
        return "C797S"
    elif "KINASE" in m:
        return "uncommon_kinase_domain"
    else:
        return "other"

def classify_mutation(row) -> str:
    text = f"{row['mutation']} {row['EGFR_type']}".upper()
    if "MISSENSE" in text:
        return "missense"
    elif "NONSENSE" in text or "STOP" in text:
        return "nonsense"
    elif "FRAMESHIFT" in text:
        return "frameshift"
    elif "SPLICE" in text:
        return "splice"
    elif "AMP" in text or "AMPLIFICATION" in text:
        return "amplification"
    elif "DEL" in text or "DELETION" in text:
        return "deletion"
    elif "SYNONYMOUS" in text or "SILENT" in text:
        return "synonymous"
    else:
        return "other"

mutation["egfr_hotspot_label"] = mutation["mutation"].apply(label_egfr_hotspot)
mutation["is_egfr_hotspot"] = mutation["egfr_hotspot_label"].ne("other").astype(int)
mutation["mutation_class"] = mutation.apply(classify_mutation, axis=1)

nonsynonymous_classes = {
    "missense", "nonsense", "frameshift", "splice", "amplification", "deletion"
}
mutation["is_nonsynonymous"] = mutation["mutation_class"].isin(nonsynonymous_classes).astype(int)

mutation_features = mutation[[
    "patient_id",
    "gene",
    "mutation",
    "EGFR_type",
    "mutation_class",
    "is_nonsynonymous",
    "is_egfr_hotspot",
    "egfr_hotspot_label",
    "cohort",
    "batch_domain"
]].copy()

print("Mutation feature shape:", mutation_features.shape)
print(mutation_features.head())

Mutation feature shape: (70, 10)
        patient_id  gene                mutation EGFR_type mutation_class  \
0  TCGA-05-4382-01  EGFR             R222L E545Q     Other          other   
1  TCGA-05-4402-01  EGFR  T751_I759delinsN I759N    Exon19       deletion   
2  TCGA-05-4410-01  EGFR                   R377S     Other          other   
3  TCGA-05-5423-01  EGFR             L833F L861Q     Other          other   
4  TCGA-17-Z026-01  EGFR                   G721V     Other          other   

   is_nonsynonymous  is_egfr_hotspot egfr_hotspot_label cohort batch_domain  
0                 0                0              other   TCGA         TCGA  
1                 1                0              other   TCGA         TCGA  
2                 0                0              other   TCGA         TCGA  
3                 0                0              other   TCGA         TCGA  
4                 0                0              other   TCGA         TCGA  


In [10]:
all_patient_ids = pd.Series(
    pd.concat([
        expression_protein_features["patient_id"],
        phospho["patient_id"],
        mutation_features["patient_id"]
    ]).dropna().unique(),
    name="patient_id"
)

master_samples = pd.DataFrame(all_patient_ids)

master_samples["has_rna"] = master_samples["patient_id"].isin(expression_protein_features["patient_id"]).astype(int)
master_samples["has_protein"] = master_samples["patient_id"].isin(expression_protein_features["patient_id"]).astype(int)
master_samples["has_phospho"] = master_samples["patient_id"].isin(phospho["patient_id"]).astype(int)
master_samples["has_mutation"] = master_samples["patient_id"].isin(mutation_features["patient_id"]).astype(int)

def infer_cohort(pid):
    return "CPTAC" if str(pid).startswith("C3L-") else "TCGA"

master_samples["cohort"] = master_samples["patient_id"].apply(infer_cohort)
master_samples["batch_domain"] = master_samples["cohort"]

print("Master sample shape:", master_samples.shape)
print(master_samples.head())

cptac_model_df.to_csv("data/cptac_model_df.csv", index=False)
mutation_features.to_csv("data/mutation_features_cleaned.csv", index=False)
master_samples.to_csv("data/master_samples.csv", index=False)

Master sample shape: (176, 7)
  patient_id  has_rna  has_protein  has_phospho  has_mutation cohort  \
0  C3L-00001        1            1            1             0  CPTAC   
1  C3L-00009        1            1            1             0  CPTAC   
2  C3L-00080        1            1            1             0  CPTAC   
3  C3L-00083        1            1            1             0  CPTAC   
4  C3L-00093        1            1            1             0  CPTAC   

  batch_domain  
0        CPTAC  
1        CPTAC  
2        CPTAC  
3        CPTAC  
4        CPTAC  


In [16]:
model_df = expression_protein_features[[
    "patient_id",
    "EGFR_RNA",
    "EGFR_PROTEIN",
    "cohort",
    "batch_domain"
]].copy()


egfr_activity_mean = full[["patient_id", "EGFR_activity_mean"]].copy()
model_df = model_df.merge(
    egfr_activity_mean,
    on="patient_id",
    how="left"
)
top_sites = ["Y1172", "Y1092", "Y1069", "Y1110", "Y1016"]

phospho_wide = (
    phospho[phospho["EGFR_binding_site"].isin(top_sites)]
    .pivot_table(
        index="patient_id",
        columns="EGFR_binding_site",
        values="EGFR_phospho_value"
    )
    .reset_index()
)

phospho_wide.columns = ["patient_id"] + [f"phospho_{c}" for c in phospho_wide.columns[1:]]

model_df = model_df.merge(phospho_wide, on="patient_id", how="left")

print(model_df.head())
print(model_df.shape)

  patient_id  EGFR_RNA  EGFR_PROTEIN cohort batch_domain  EGFR_activity_mean  \
0  C3L-00001     17.07     28.192875  CPTAC        CPTAC                   1   
1  C3L-00001     17.07     28.192875  CPTAC        CPTAC                   1   
2  C3L-00009     11.86     25.219585  CPTAC        CPTAC                   1   
3  C3L-00009     11.86     25.219585  CPTAC        CPTAC                   1   
4  C3L-00080     12.58     25.238803  CPTAC        CPTAC                   1   

   phospho_Y1016  phospho_Y1069  phospho_Y1092  phospho_Y1110  phospho_Y1172  
0            NaN            NaN            NaN            NaN      20.075879  
1            NaN            NaN            NaN            NaN      20.075879  
2            NaN            NaN      16.931486            NaN      18.767056  
3            NaN            NaN      16.931486            NaN      18.767056  
4            NaN            NaN            NaN            NaN      18.026248  
(409, 11)
